# 03. Raspberry Pi 5에서 HEF 로드와 추론 확인 (HailoRT Python API)

- 셀 1: picamera2로 카메라 모듈(imx708) 인식을 확인합니다.
- 셀 3: 컴파일한 HEF를 PCIe로 연결된 Hailo-8L에 올리고, 더미 입력으로 추론해 입출력 스트림 형태와 NMS 후처리 출력 형식을 확인합니다.

아래 출력은 실제 실행 결과를 그대로 남긴 것입니다.


In [ ]:
import cv2
from picamera2 import Picamera2

picam = Picamera2()
picam.configure(
    picam.create_preview_configuration(
        main = {
            "format": 'XRGB8888',
            'size': (1024, 768)
        }
    )
)

# picam.start()

[0:05:58.334591342] [2671]  INFO Camera camera_manager.cpp:325 libcamera v0.3.2+27-7330f29b
[0:05:58.364325639] [2719]  INFO RPI pisp.cpp:695 libpisp version v1.0.7 28196ed6edcf 29-08-2024 (16:33:32)
[0:05:58.375980255] [2719]  INFO RPI pisp.cpp:1154 Registered camera /base/axi/pcie@120000/rp1/i2c@88000/imx708@1a to CFE device /dev/media2 and ISP device /dev/media0 using PiSP variant BCM2712_C0
[0:05:58.380619030] [2671]  INFO Camera camera.cpp:1197 configuring streams: (0) 1024x768-XRGB8888 (1) 1536x864-BGGR_PISP_COMP1
[0:05:58.380747200] [2719]  INFO RPI pisp.cpp:1450 Sensor: /base/axi/pcie@120000/rp1/i2c@88000/imx708@1a - Selected sensor format: 1536x864-SBGGR10_1X10 - Selected CFE format: 1536x864-PC1B


In [ ]:
import numpy as np

from multiprocessing import Process
from hailo_platform import (
    HEF, VDevice, HailoStreamInterface, 
    ConfigureParams, InputVStreamParams, OutputVStreamParams,
    InputVStreams, InferVStreams, OutputVStreams,
    FormatType
)

In [ ]:
# Loading compiled HEFs to device:
import sys

hef_path = './yolov8n_float32.hef'
hef = HEF(hef_path)
    
# Configure network groups
configure_params = ConfigureParams.create_from_hef(hef=hef, interface=HailoStreamInterface.PCIe)

with VDevice() as target: # why use 'with - as' keyword -> monopoly of resource 
    print(target)
    network_groups = target.configure(hef, configure_params)
    network_group = network_groups[0]
    network_group_params = network_group.create_params()
    # print('network_groups:')
    # print(network_groups)
    # print('network_group:')
    # print(network_group)
    # print('network_group_params:')
    # print(network_group_params)
    input_vstream_params = InputVStreamParams.make(
        network_group,
        format_type = FormatType.FLOAT32
    )
    output_vstream_params = OutputVStreamParams.make(
        network_group,
        format_type = FormatType.FLOAT32
    )
    # print("input_vstream_params:")
    # print(input_vstream_params)
    # print("output_vstream_params:")
    # print(output_vstream_params)

    # SHAPE FORMAT: HWC
    input_vstream_info = hef.get_input_stream_infos()
    for info_input_elem in input_vstream_info: 
        print(f"{info_input_elem} SHAPE: {info_input_elem.shape}")

    height_input, width_input, channel_input = input_vstream_info[0].shape
        
    output_vstream_info = hef.get_output_stream_infos()
    for info_output_elem in output_vstream_info: 
        print(f"{info_output_elem} SHAPE: {info_output_elem.shape}")

    # Generate random input dataset (dummy for inference test)
    low, high = 2, 20
    num_of_img = 1
    dataset = np.random.randint(
        low, high, (
            num_of_img, height_input, width_input, channel_input
        )
    ).astype(np.float32)

    with InferVStreams(network_group, input_vstream_params, output_vstream_params) as infer_pipeline:
        input_layer_info = input_vstream_info[0]
        input_data = { input_layer_info.name: dataset }
        with network_group.activate(network_group_params = network_group_params):
            infer_results = infer_pipeline.infer(input_data) # output of inference
            # format of Hailo NMS output
            # Image(Batch), Class(if element is none, ignore), np.array(
            #   # of Detect, (y_min, x_min, y_max, x_max, score)
            # )
            print(len(infer_results['yolov8n_float32/yolov8_nms_postprocess']))
            for idx in range(len(infer_results['yolov8n_float32/yolov8_nms_postprocess'])):
                print(len(infer_results['yolov8n_float32/yolov8_nms_postprocess'][0]))
                for iidx in range(len(infer_results['yolov8n_float32/yolov8_nms_postprocess'][idx])):
                    print(infer_results['yolov8n_float32/yolov8_nms_postprocess'][idx][iidx])

            



StreamInfo("yolov8n_float32/input_layer1") SHAPE: (640, 640, 3)
StreamInfo("yolov8n_float32/conv53_97") SHAPE: (40, 40, 80)
StreamInfo("yolov8n_float32/conv41_97") SHAPE: (80, 80, 64)
StreamInfo("yolov8n_float32/conv52_97") SHAPE: (40, 40, 64)
StreamInfo("yolov8n_float32/conv42_97") SHAPE: (80, 80, 80)
StreamInfo("yolov8n_float32/conv63_97") SHAPE: (20, 24, 80)
StreamInfo("yolov8n_float32/conv62_97") SHAPE: (20, 24, 64)
1
80
[[0.00236964 0.00373191 0.9988998  0.99564517 0.4614572 ]]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[[-0.00187734  0.00558776  0.99923056  0.9989428   0.2510327 ]]
[]
[[0.00153232 0.00616145 0.9921353  1.000316   0.33594084]]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
